# 🌾 Godown Spoilage Classifier — Colab Training

Run this notebook on **Google Colab with GPU** (Runtime → Change runtime type → GPU).

**Steps:**
1. Run cells in order
2. Download `spoilage_model.h5` at the end
3. Copy it to `backend/model/spoilage_model.h5` in your local project
4. Restart the backend server — it will load the real model automatically

In [1]:
# ── Cell 1: Install / verify TensorFlow ───────────────────────────────────────
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPU available:', tf.test.is_gpu_available())

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# ── Cell 2: Generate synthetic dataset (same script as local) ─────────────────
import random
import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw, ImageFilter
import os

CLASS_CONFIGS = {
    'healthy':       {'bg': (34,139,34),   'spot': (50,205,50),   'noise': 15, 'spots': 0,  'blur': 0.5},
    'mold':          {'bg': (80,60,30),    'spot': (180,180,50),  'noise': 30, 'spots': 20, 'blur': 1.5},
    'pest_damage':   {'bg': (139,90,43),   'spot': (30,20,10),    'noise': 20, 'spots': 12, 'blur': 0.3},
    'discoloration': {'bg': (180,150,60),  'spot': (200,100,40),  'noise': 25, 'spots': 8,  'blur': 0.8},
}
rng = random.Random(42)
np.random.seed(42)

def var(c, v=25): return tuple(max(0,min(255,x+rng.randint(-v,v))) for x in c)

def make_img(cls):
    cfg = CLASS_CONFIGS[cls]
    arr = np.full((224,224,3), var(cfg['bg'],20), dtype=np.uint8)
    n = rng.randint(cfg['noise']//2, cfg['noise'])
    arr = np.clip(arr.astype(np.int16) + np.random.randint(-n,n+1,arr.shape),0,255).astype(np.uint8)
    img = Image.fromarray(arr,'RGB')
    draw = ImageDraw.Draw(img)
    for _ in range(cfg['spots']):
        x,y=rng.randint(15,209),rng.randint(15,209)
        r=rng.randint(5,20)
        draw.ellipse([x-r,y-r,x+r,y+r],fill=var(cfg['spot'],30))
    return img.filter(ImageFilter.GaussianBlur(cfg['blur']))

for split,n in [('train',300),('val',75)]:
    for cls in CLASS_CONFIGS:
        p = Path(f'dataset/{split}/{cls}')
        p.mkdir(parents=True, exist_ok=True)
        for i in range(n):
            make_img(cls).save(p/f'{cls}_{i:04d}.jpg', quality=85)
        print(f'  {split}/{cls}: {n} images')

print('\n✅ Dataset generated!')

In [ ]:
# ── Cell 3: Build and train model ─────────────────────────────────────────────
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

IMG_SIZE = (224, 224)
BATCH    = 32
EPOCHS   = 12
CLASSES  = ['healthy', 'mold', 'pest_damage', 'discoloration']

opts = dict(image_size=IMG_SIZE, batch_size=BATCH, label_mode='categorical', class_names=CLASSES)
normalize = layers.Rescaling(1.0/255)
train_ds = tf.keras.utils.image_dataset_from_directory('dataset/train', **opts)
val_ds   = tf.keras.utils.image_dataset_from_directory('dataset/val',   **opts)
train_ds = train_ds.map(lambda x,y:(normalize(x),y)).cache().shuffle(1000).prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds.map(lambda x,y:(normalize(x),y)).cache().prefetch(tf.data.AUTOTUNE)

base = MobileNetV2(input_shape=(224,224,3), include_top=False, weights='imagenet')
base.trainable = False

model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(CLASSES), activation='softmax'),
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5),
]
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)

model.save('spoilage_model.h5')
print('\n✅ Saved spoilage_model.h5')

In [ ]:
# ── Cell 4: Download the model ─────────────────────────────────────────────────
from google.colab import files
files.download('spoilage_model.h5')
print('Copy spoilage_model.h5 → backend/model/spoilage_model.h5 in your project')